### **📸 Snapshots in dbt**

#### **1️⃣ What problem do snapshots solve?**

In real databases, **rows change.**

Examples:

- Host changes name

- Customer updates email

- Employee status becomes inactive

- Account is deleted


**❌ Without snapshots**

When data changes, the **old value is lost forever.**

**✅ With snapshots**

You can answer questions like:

- What was the value **before**?

- **When** did it change?

- **How long** was a value valid?

- What did the data look like **on a specific date?**

**👉 Snapshots convert mutable tables into historical tables**

-------------

#### **2️⃣ What is a snapshot in dbt?**

> **A snapshot is a dbt object that tracks historical changes of rows over time by storing multiple versions of the same record.**

In simpler words:

> **Snapshots implement Slowly Changing Dimensions (SCD Type 2).**

----------------

#### **3️⃣ When should snapshots be used?**

**✅ Use snapshots for:**

- Dimension tables

- Entity-like data

- Tables that get **UPDATED or DELETED**

Examples:

- hosts

- customers

- employees

- subscriptions

- accounts

**❌ Do NOT use snapshots for:**

- Event / fact tables (orders, clicks, logs)

- Append-only data

- High-frequency metrics

------------------

#### **4️⃣ Where snapshots live in a dbt project**

In [ ]:
snapshots/
  scd_raw_hosts.yml

📌 Snapshots **do NOT go inside** `models/`

📌 They have their **own execution command**

----------------

#### **5️⃣ Snapshot execution**

In [ ]:
dbt snapshot

Important:

- ❌ `dbt run` → does NOT run snapshots

- ❌ `dbt build` → only runs snapshots if explicitly included

Snapshots are **explicitly controlled**

----------------------


#### **6️⃣ Two ways to define snapshots in dbt**

dbt supports:

- **SQL-based snapshots** (older, more verbose)

- **YAML-based snapshots** (newer, cleaner)

We use **YAML-based snapshots ✅**

This is **perfectly valid and recommended**

----------------

#### **7️⃣ Your snapshot definition (reference)**

In [ ]:
snapshots: 
  - name: scd_raw_hosts 
    relation: source('airbnb', 'hosts')
    config:
      unique_key: id 
      updated_at: updated_at 
      strategy: timestamp 
      hard_deletes: invalidate 

Everything below is explained using this exact example.

-------------

#### **8️⃣ How dbt interprets a snapshot (mental model)**

When you run:

In [ ]:
dbt snapshot

dbt does this:

- Reads snapshot definitions

- Queries the **current source data**

- Compares it with **previous snapshot state**

- Detects:

    - New rows

    - Updated rows

    - Deleted rows

- Inserts new versions where needed

- Closes old versions by setting `dbt_valid_to`

--------------------

#### **9️⃣ Line-by-line explanation (very important)**

**🔹 snapshots:**

In [ ]:
snapshots:

Top-level key telling dbt:

> “This file contains snapshot definitions.”

Mandatory.

------------------

**🔹 name: scd_raw_hosts**

In [ ]:
- name: scd_raw_hosts

- Snapshot name

- Also becomes the **table name** in the database

Example in Snowflake:

In [ ]:
  DEV_SNAPSHOTS.SCD_RAW_HOSTS

----------


**🔹 relation: source('airbnb', 'hosts')**

In [ ]:
relation: source('airbnb', 'hosts')

Meaning:

> “Track changes from the `hosts` source table under the `airbnb` source.”

Internally resolves to something like:

In [ ]:
SELECT * FROM RAW.HOSTS

✅ Uses `source()` → best practice

❌ Never hardcode schema/table names

----------

**🔹 unique_key: id**

In [ ]:
unique_key: id

This is the **most critical field.**

Meaning:

> “Each row represents one host, uniquely identified by id.”

dbt uses this to:

- Match rows across runs

- Decide whether a row changed or not

📌 Same `id` → same real-world entity

If this is wrong → snapshot history is wrong.

-----------------

#### **🔟 Snapshot strategies (how changes are detected)**

dbt supports two strategies:

**🕒 Strategy 1: `timestamp` (you used this)**

In [ ]:
strategy: timestamp
updated_at: updated_at

**How it works:**

- dbt compares the `updated_at` value

- If it `increases`, dbt assumes the row changed

**Requirements:**

- `updated_at` column must exist

- It must update **every time any tracked column changes**

**Example:**

| id | host_name | updated_at |
| -- | --------- | ---------- |
| 1  | Alex      | 2024-01-01 |
| 1  | Alexander | 2024-01-05 |

→ dbt creates a **new snapshot row**

------------------

#### **🔍 Strategy 2: check (for understanding)**


In [ ]:
strategy: check
check_cols: ['host_name', 'is_superhost']

dbt compares column values directly.

Change in ANY column → new version.

Used when:

- No reliable timestamp exists

-----------------


#### **1️⃣1️⃣ Handling deletes (`hard_deletes`**)

In [ ]:
hard_deletes: invalidate

This controls what happens when a row is **physically deleted** from the source.

---------------

**What is a hard delete?**

In [ ]:
DELETE FROM hosts WHERE id = 101;

Row is **gone** from source.

---------------

**`invalidate` means:**

> “If a row disappears, close the snapshot record.”

dbt sets:

In [ ]:
dbt_valid_to = <deletion time>

📌 History is preserved

📌 Analytics remain accurate

-----------------

**Why this matters**

Without `invalidate`:

- Deleted entities may appear as “still active”

- Historical reporting breaks


-------------------


#### **1️⃣2️⃣ Columns dbt automatically adds**

You **do not define these**, dbt adds them:


| Column           | Meaning                   |
| ---------------- | ------------------------- |
| `dbt_scd_id`     | Unique version identifier |
| `dbt_updated_at` | Snapshot run time         |
| `dbt_valid_from` | When this version started |
| `dbt_valid_to`   | When this version ended   |


**Important rule:**

- **Current version** → `dbt_valid_to IS NULL`


--------------------

**1️⃣3️⃣ What happens during snapshot runs (timeline)**

**First run**

- Snapshot table is empty

- All rows are inserted

- dbt_valid_to = NULL


-----------------

**Update happens**

- dbt closes old version

- Inserts new version

-------------

**Delete happens**

- dbt invalidates row

- No new row inserted

- History preserved

--------------

#### **1️⃣4️⃣ Example snapshot output**

| id | host_name | dbt_valid_from | dbt_valid_to |
| -- | --------- | -------------- | ------------ |
| 1  | Alex      | 2024-01-01     | 2024-01-05   |
| 1  | Alexander | 2024-01-05     | NULL         |



----------

**1️⃣5️⃣ Querying snapshots correctly (very important)**

**✅ Current state only**

In [ ]:
SELECT *
FROM {{ ref('scd_raw_hosts') }}
WHERE dbt_valid_to IS NULL

**🕰 Point-in-time analysis**

In [ ]:
SELECT *
FROM {{ ref('scd_raw_hosts') }}
WHERE '2024-02-01'
BETWEEN dbt_valid_from
AND COALESCE(dbt_valid_to, CURRENT_DATE)

#### **1️⃣6️⃣ Snapshots vs models vs incremental models**

| Feature        | Models  | Incremental | Snapshots    |
| -------------- | ------- | ----------- | ------------ |
| Latest state   | ✅       | ✅           | ❌            |
| History        | ❌       | ❌           | ✅            |
| Tracks updates | ❌       | ❌           | ✅            |
| Tracks deletes | ❌       | ❌           | ✅            |
| Execution      | dbt run | dbt run     | dbt snapshot |


------------

#### **1️⃣7️⃣ Common mistakes beginners make**

❌ Expecting snapshots to auto-run

❌ Wrong unique_key

❌ updated_at not updating

❌ Using snapshots for fact tables

❌ Forgetting dbt_valid_to IS NULL filter

-------------

#### **1️⃣8️⃣ Best practices (must remember)**

✅ Snapshot only **dimensions**

✅ Prefer `timestamp` strategy

✅ Keep snapshot schemas separate

✅ Always document snapshots

✅ Build final dimensions **on top of snapshots**

✅ Use snapshots only when history is required
